# 12 · SC-FEGAN Pretrained Inference (실제 inpainting 결과)

**목적**: SC-FEGAN의 pretrained ckpt로 검은 마스크 → **변형된 코/눈/턱** 복원.

**파이프라인**:
```
사진 → 랜드마크 → 9채널 입력 → SC-FEGAN inference → Refinement → 최종
```

**SC-FEGAN API 분석 결과** (model.py 코드 기반):
- batch 형식: `(B, H, W, 9)` = `[images(3) + sketch(1) + color(3) + mask(1) + noise(1)]`
- 호출: `model.demo(config, batch)` → 출력 (B, H, W, 3)
- 출력 의미: `gen_img * mask + incomplete_img` (mask 영역만 변형)

**환경**: Colab T4 + TF 2.x compat mode. SC-FEGAN의 tf.contrib 의존성은 패치 셀에서 제거.

**산출물**: `/MyDrive/cv-final-ckpts/samples/inference_{procedure}.png`

## 1. 환경 셋업 + repo clone

In [ ]:
import os, sys, subprocess
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        from google.colab import userdata
        token = userdata.get('GH_TOKEN').strip()
        result = subprocess.run(
            ['git', 'clone', f'https://{token}@github.com/keonU206/cv-final.git', REPO],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f'cv-final clone 실패: {result.stderr}')
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    
    SCFEGAN_REPO = '/content/SC-FEGAN'
    if not os.path.exists(SCFEGAN_REPO):
        !git clone https://github.com/run-youngjoo/SC-FEGAN.git $SCFEGAN_REPO
    
    os.environ['TF_USE_LEGACY_KERAS'] = '1'
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
    !pip install -q mediapipe==0.10.21 opencv-python pyyaml 'tensorflow<2.16' tf-keras segmentation_models_pytorch

print(f'CWD: {os.getcwd()}')
print(f'SC-FEGAN: {SCFEGAN_REPO}')

## 2. SC-FEGAN 코드 패치 (TF 2.x 호환)

원본 SC-FEGAN은 TF 1.15 + `tf.contrib`에 의존. TF 2.x compat 환경에서는 동작 안 함.

패치 사항:
- `model.py`: `import tensorflow.contrib.slim.nets` 제거 (안 쓰임)
- `model.py`: `tf.contrib.framework.load_variable` → `tf.train.load_variable`
- `ops.py`: `tf.contrib.framework.python.ops.add_arg_scope` → no-op decorator

In [ ]:
def patch_scfegan_files():
    """SC-FEGAN repo의 tf.contrib 의존성 제거 패치."""
    
    # 1. model.py 패치
    model_py = os.path.join(SCFEGAN_REPO, 'model.py')
    with open(model_py) as f:
        content = f.read()
    
    patches = [
        # contrib.slim.nets 제거 (안 쓰이는 dead import)
        ('import tensorflow.contrib.slim.nets as nets',
         '# import tensorflow.contrib.slim.nets as nets  # TF 2.x 호환 패치: 제거'),
        # load_variable 대체
        ('tf.contrib.framework.load_variable(ckpt_path, from_name)',
         'tf.train.load_variable(ckpt_path, from_name)'),
    ]
    
    for old, new in patches:
        if old in content:
            content = content.replace(old, new)
            print(f'  ✅ model.py 패치: {old[:50]}...')
        else:
            print(f'  ⚠ model.py에 없음 (이미 패치되었을 수 있음): {old[:50]}...')
    
    with open(model_py, 'w') as f:
        f.write(content)
    
    # 2. ops.py 패치 — add_arg_scope를 no-op decorator로 대체
    ops_py = os.path.join(SCFEGAN_REPO, 'ops.py')
    with open(ops_py) as f:
        content = f.read()
    
    contrib_import = 'from tensorflow.contrib.framework.python.ops import add_arg_scope'
    no_op_decorator = '''# TF 2.x 호환 패치: contrib 제거, no-op decorator로 대체
def add_arg_scope(func):
    """No-op decorator — SC-FEGAN inference엔 의미 없음."""
    return func'''
    
    if contrib_import in content:
        content = content.replace(contrib_import, no_op_decorator)
        print(f'  ✅ ops.py 패치: add_arg_scope 대체')
    else:
        print(f'  ⚠ ops.py에 contrib import 없음 (이미 패치되었을 수 있음)')
    
    with open(ops_py, 'w') as f:
        f.write(content)
    
    print('\n패치 완료. 첫 줄 확인:')
    print(f'  model.py: {open(model_py).readline().strip()}')
    print(f'  ops.py:   {open(ops_py).readline().strip()}')

patch_scfegan_files()

## 3. TF compat 활성화 + Model 클래스 import

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
print(f'TF: {tf.__version__}')

import sys
if SCFEGAN_REPO not in sys.path:
    sys.path.insert(0, SCFEGAN_REPO)

from model import Model
print(f'✅ Model 클래스 import 성공: {Model.__module__}.{Model.__name__}')

## 4. SC-FEGAN 모델 빌드 + ckpt 로드

Config 클래스 대신 간단한 객체로 대체 (yaml 의존성 제거).

In [ ]:
# Config 대체: SC-FEGAN이 기대하는 속성만 가진 가벼운 객체
class SimpleConfig:
    INPUT_SIZE = 512
    BATCH_SIZE = 1
    CKPT_DIR = '/content/drive/MyDrive/SC-FEGAN-ckpt/SC-FEGAN.ckpt'

config = SimpleConfig()
print(f'ckpt 경로 존재: {os.path.exists(config.CKPT_DIR + ".index")}')

# Graph 빌드 + ckpt 로드
tf.reset_default_graph()
model = Model(config)
print('✅ Model 인스턴스 생성')

model.load_demo_graph(config)  # 자동으로 sess 생성 + ckpt 로드 + warmup
print('✅ ckpt 로드 + warmup 완료')
print(f'  attributes: images, sketches, color, masks, noises, demo_output')

## 5. 사진 업로드 + 랜드마크 추출

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path

SIZE = 512
OUTPUT_DIR = Path('outputs/scfegan_inference')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from google.colab import files
    print('얼굴 사진 1장 업로드:')
    uploaded = files.upload()
    IMG_PATH = list(uploaded.keys())[0]
else:
    IMG_PATH = 'demo_face.jpg'

image_rgb = cv2.cvtColor(cv2.imread(IMG_PATH), cv2.COLOR_BGR2RGB)
image_rgb = cv2.resize(image_rgb, (SIZE, SIZE))

import mediapipe as mp
with mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True, max_num_faces=1, refine_landmarks=True,
) as fm:
    res = fm.process(image_rgb)

if not res.multi_face_landmarks:
    raise RuntimeError('얼굴 감지 실패')

lms = res.multi_face_landmarks[0].landmark
landmarks = np.array([[lm.x * SIZE, lm.y * SIZE] for lm in lms], dtype=np.int32)
print(f'✅ Landmarks: {landmarks.shape}')

plt.figure(figsize=(5, 5))
plt.imshow(image_rgb); plt.axis('off'); plt.title('Input')
plt.show()

## 6. 시술별 9채널 입력 생성

⚠ SC-FEGAN demo() 함수는 batch[:,:,:,:3]을 **원본 image**로 받음 (내부에서 mask 적용).
→ `to_scfegan_tensor`의 첫 3채널을 incomplete가 아닌 **원본**으로 교체.

In [ ]:
from project.input_generator import compose_scfegan_input

PROCEDURES = ['nose_tip', 'double_eyelid', 'jaw_v_line']
composed_all = {}
batches = {}

for proc_id in PROCEDURES:
    composed = compose_scfegan_input(
        image_rgb, landmarks, proc_id,
        size=SIZE, intensity=0.6, seed=42,
    )
    composed_all[proc_id] = composed
    
    # SC-FEGAN demo() 형식: [원본 image(3) + sketch(1) + color(3) + mask(1) + noise(1)]
    image_n = (image_rgb.astype(np.float32) / 127.5) - 1.0  # [-1, 1]
    color_n = (composed['color'].astype(np.float32) / 127.5) - 1.0
    sketch_n = composed['sketch'].astype(np.float32) / 255.0
    mask_n = composed['mask'].astype(np.float32) / 255.0
    noise_n = composed['noise']
    
    batch = np.concatenate(
        [image_n, sketch_n, color_n, mask_n, noise_n], axis=-1
    )[None, ...].astype(np.float32)  # (1, H, W, 9)
    batches[proc_id] = batch
    print(f'{proc_id}: batch shape {batch.shape}, range [{batch.min():.2f}, {batch.max():.2f}]')

## 7. SC-FEGAN Inference (실제 inpainting!)

In [ ]:
results_scfegan = {}

for proc_id, batch in batches.items():
    print(f'\n{proc_id} inference 중...')
    try:
        # model.demo(config, batch) → (1, H, W, 3) in [-1, 1]
        output = model.demo(config, batch)
        # [-1, 1] → [0, 255]
        result = ((output[0] + 1) * 127.5).clip(0, 255).astype(np.uint8)
        results_scfegan[proc_id] = result
        print(f'  ✅ shape: {result.shape}, range: [{result.min()}, {result.max()}]')
    except Exception as e:
        import traceback
        print(f'  ❌ {type(e).__name__}: {e}')
        traceback.print_exc()
        results_scfegan[proc_id] = composed_all[proc_id]['incomplete_image']

## 8. Refinement 적용 (학습된 ckpt로 정제)

In [ ]:
import torch
from project.refinement.model import build_refinement_wrapper

device = 'cuda' if torch.cuda.is_available() else 'cpu'
REF_CKPT = '/content/drive/MyDrive/cv-final-ckpts/refinement_best.pt'

if os.path.exists(REF_CKPT):
    ref_model = build_refinement_wrapper(encoder_weights=None).to(device)
    ref_model.load_state_dict(torch.load(REF_CKPT, map_location=device))
    ref_model.eval()
    print(f'✅ Refinement 로드: {REF_CKPT}')
else:
    ref_model = None
    print('⚠ Refinement ckpt 없음 → SC-FEGAN 결과 그대로 사용')

def refine(img_rgb):
    if ref_model is None:
        return img_rgb
    tensor = torch.from_numpy(img_rgb).float() / 127.5 - 1.0
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        out = ref_model(tensor)[0].cpu()
    return ((out.clamp(-1, 1) + 1) * 127.5).byte().permute(1, 2, 0).numpy()

results_final = {}
for proc_id, sc_out in results_scfegan.items():
    results_final[proc_id] = refine(sc_out)
    print(f'  {proc_id}: refined')

## 9. 결과 시각화 — 4컬럼 비교

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
for i, proc_id in enumerate(PROCEDURES):
    composed = composed_all[proc_id]
    
    axes[i, 0].imshow(image_rgb)
    axes[i, 0].set_title(f'{proc_id}\nBefore (원본)')
    axes[i, 1].imshow(composed['incomplete_image'])
    axes[i, 1].set_title('Incomplete\n(SC-FEGAN 입력)')
    axes[i, 2].imshow(results_scfegan[proc_id])
    axes[i, 2].set_title('SC-FEGAN 출력 ⭐\n(inpainted)')
    axes[i, 3].imshow(results_final[proc_id])
    axes[i, 3].set_title('+ Refinement\n(최종)')
    for ax in axes[i]:
        ax.axis('off')
    
    # 개별 저장
    cv2.imwrite(
        str(OUTPUT_DIR / f'final_{proc_id}.png'),
        cv2.cvtColor(results_final[proc_id], cv2.COLOR_RGB2BGR),
    )
    cv2.imwrite(
        str(OUTPUT_DIR / f'scfegan_only_{proc_id}.png'),
        cv2.cvtColor(results_scfegan[proc_id], cv2.COLOR_RGB2BGR),
    )

plt.tight_layout()
save_path = OUTPUT_DIR / 'all_results.png'
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 저장: {save_path}')

## 10. Drive 백업

In [ ]:
if IS_COLAB:
    DRIVE_OUT = Path('/content/drive/MyDrive/cv-final-ckpts/samples')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    !cp -r {OUTPUT_DIR}/* {DRIVE_OUT}/
    print(f'✅ Drive 백업: {DRIVE_OUT}/')
    !ls -lh {DRIVE_OUT}/

print('\n=== Phase 7-E 완료 ===')
print('이 결과를:')
print('  1. 발표 슬라이드에 추가 (final_*.png)')
print('  2. Streamlit에서 정적 이미지로 표시')
print('  3. PDF 견적서의 Before/After 영역에 활용')